# Experiment 12 — Rao local activation features + XGBoost

Two patient-level XGBoost variants are evaluated with the same 577-patient cohort and the same shared 10 folds:

- **TN12-A:** all 93 local `%A` features
- **TN12-B:** the 62 local features listed as significant in Rao Table 2

This is an extension of Rao, not a pure classifier-only replacement: Rao trains one univariate Bayesian classifier per feature, whereas XGBoost combines multiple features simultaneously.


## 1. Install

In [ ]:
# !pip install numpy pandas scikit-learn xgboost


## 2. Imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    matthews_corrcoef, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier


## 3. Configuration

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 10
EXPECTED_PATIENTS = 577
EXPECTED_MGMT_1 = 301
EXPECTED_MGMT_0 = 276

LOCAL_FEATURES_CSV = Path("outputs_rao2021/local_93_percent_activation.csv")

SHARED_DIR = Path("shared_rao_comparison")
SHARED_FOLD_CSV = SHARED_DIR / "mgmt_577_fold_assignment_10fold.csv"

OUTDIR = Path("outputs_exp12_rao_features_xgboost")
OUTDIR.mkdir(parents=True, exist_ok=True)

WEIGHT_CONDITIONS = ["None", "Balanced"]

RAO_LOCAL_62 = ['glcm_DifferenceEntropy', 'firstorder_Skewness', 'firstorder_Minimum', 'firstorder_Range', 'glcm_Imc1', 'firstorder_Kurtosis', 'glcm_JointAverage', 'glcm_SumAverage', 'glcm_Autocorrelation', 'firstorder_Entropy', 'glcm_DifferenceVariance', 'firstorder_Uniformity', 'glcm_ClusterProminence', 'glcm_ClusterTendency', 'firstorder_Variance', 'firstorder_MeanAbsoluteDeviation', 'firstorder_RobustMeanAbsoluteDeviation', 'firstorder_InterquartileRange', 'glcm_Imc2', 'glcm_MCC', 'firstorder_Maximum', 'glcm_Idm', 'glcm_Id', 'glcm_ClusterShade', 'glcm_Contrast', 'glcm_InverseVariance', 'glcm_DifferenceAverage', 'firstorder_90Percentile', 'firstorder_Median', 'firstorder_RootMeanSquared', 'firstorder_Mean', 'glszm_HighGrayLevelZoneEmphasis', 'glrlm_LongRunEmphasis', 'gldm_SmallDependenceHighGrayLevelEmphasis', 'glrlm_GrayLevelNonUniformityNormalized', 'glszm_SmallAreaHighGrayLevelEmphasis', 'gldm_GrayLevelVariance', 'ngtdm_Complexity', 'ngtdm_Strength', 'glrlm_RunPercentage', 'gldm_DependenceNonUniformityNormalized', 'glszm_LargeAreaLowGrayLevelEmphasis', 'glrlm_LongRunHighGrayLevelEmphasis', 'glrlm_RunVariance', 'glrlm_ShortRunHighGrayLevelEmphasis', 'glrlm_RunLengthNonUniformityNormalized', 'gldm_LargeDependenceEmphasis', 'glrlm_GrayLevelVariance', 'gldm_DependenceVariance', 'glszm_ZonePercentage', 'ngtdm_Contrast', 'gldm_HighGrayLevelEmphasis', 'glszm_SmallAreaEmphasis', 'glszm_SizeZoneNonUniformityNormalized', 'glrlm_ShortRunEmphasis', 'glszm_GrayLevelNonUniformityNormalized', 'ngtdm_Busyness', 'glszm_GrayLevelVariance', 'glcm_SumSquares', 'gldm_SmallDependenceEmphasis', 'glrlm_HighGrayLevelRunEmphasis', 'gldm_LargeDependenceLowGrayLevelEmphasis']
assert len(RAO_LOCAL_62) == 62


## 4. Load Rao local features

In [ ]:
df = pd.read_csv(LOCAL_FEATURES_CSV)

required = {"patient_id", "mgmt"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

df["patient_id"] = df["patient_id"].astype(str)
df["mgmt"] = df["mgmt"].astype(int)
df = df.sort_values("patient_id").reset_index(drop=True)

if len(df) != 577 or df["patient_id"].nunique() != 577:
    raise ValueError("TN12 requires exactly 577 unique patients.")

counts = df["mgmt"].value_counts().to_dict()
if counts.get(1, 0) != EXPECTED_MGMT_1 or counts.get(0, 0) != EXPECTED_MGMT_0:
    raise ValueError(f"Unexpected MGMT distribution: {counts}")

feature_cols_93 = [
    c for c in df.columns
    if c not in {"patient_id", "mgmt"}
    and pd.api.types.is_numeric_dtype(df[c])
]

if len(feature_cols_93) != 93:
    raise ValueError(f"Expected 93 local activation features, found {len(feature_cols_93)}.")

missing62 = sorted(set(RAO_LOCAL_62) - set(feature_cols_93))
if missing62:
    raise ValueError(f"Missing Rao Table-2 features: {missing62}")

matrix = df[feature_cols_93].to_numpy(dtype=float)
bad = ~np.isfinite(matrix)
if bad.any():
    bad_patients = df.loc[bad.any(axis=1), "patient_id"].tolist()
    raise ValueError(
        "NaN/Inf detected; dropping patients is forbidden. "
        f"Affected patients (first 10): {bad_patients[:10]}"
    )

FEATURE_SETS = {
    "TN12A_All93": feature_cols_93,
    "TN12B_Rao62": RAO_LOCAL_62,
}

print("Patients:", len(df))
print("Feature sets:", {k: len(v) for k, v in FEATURE_SETS.items()})
print(df["mgmt"].value_counts().sort_index())
display(df.head())


## 5. Create/reuse identical 10-fold assignments

In [ ]:
def load_shared_folds(cohort):
    if not SHARED_FOLD_CSV.exists():
        raise FileNotFoundError(
            f"{SHARED_FOLD_CSV} not found. Run Experiment 11 first."
        )

    folds = pd.read_csv(SHARED_FOLD_CSV)
    folds["patient_id"] = folds["patient_id"].astype(str)

    required = {"patient_id", "mgmt", "fold"}
    if required - set(folds.columns):
        raise ValueError("Invalid shared fold schema.")
    if len(folds) != 577 or folds["patient_id"].nunique() != 577:
        raise ValueError("Shared folds must contain 577 unique patients.")

    expected = cohort[["patient_id", "mgmt"]]
    merged = expected.merge(
        folds[["patient_id", "mgmt", "fold"]],
        on=["patient_id", "mgmt"],
        how="outer",
        indicator=True,
    )
    if not (merged["_merge"] == "both").all():
        raise ValueError("Shared fold assignment does not match the current cohort.")

    return folds[["patient_id", "fold"]]


folds = load_shared_folds(df)
work = df.merge(folds, on="patient_id", how="inner", validate="one_to_one")
print(work.groupby(["fold", "mgmt"]).size().unstack(fill_value=0))


## 6. Thesis XGBoost configuration

In [ ]:
def build_xgb():
    return XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        min_child_weight=1,
        gamma=0,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0,
        reg_lambda=1,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


## 7. Run 10-fold CV

In [ ]:
def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fp) if (tn + fp) else np.nan


rows = []

for feature_set_name, selected_features in FEATURE_SETS.items():
    for weight_condition in WEIGHT_CONDITIONS:
        for fold in range(1, N_SPLITS + 1):
            train = work[work["fold"] != fold]
            valid = work[work["fold"] == fold]

            Xtr = train[selected_features].to_numpy(dtype=np.float32)
            ytr = train["mgmt"].to_numpy(dtype=int)
            Xva = valid[selected_features].to_numpy(dtype=np.float32)
            yva = valid["mgmt"].to_numpy(dtype=int)

            sample_weight = None
            if weight_condition == "Balanced":
                classes = np.array([0, 1])
                cw_vals = compute_class_weight("balanced", classes=classes, y=ytr)
                cw = {0: float(cw_vals[0]), 1: float(cw_vals[1])}
                sample_weight = np.array([cw[int(v)] for v in ytr], dtype=np.float32)

            model = build_xgb()
            model.fit(Xtr, ytr, sample_weight=sample_weight)

            prob = model.predict_proba(Xva)[:, 1]
            pred = (prob >= 0.5).astype(int)

            rows.append({
                "feature_set": feature_set_name,
                "n_features": len(selected_features),
                "condition": weight_condition,
                "fold": fold,
                "n_train": len(train),
                "n_valid": len(valid),
                "accuracy": accuracy_score(yva, pred),
                "precision": precision_score(yva, pred, zero_division=0),
                "recall": recall_score(yva, pred, zero_division=0),
                "specificity": specificity(yva, pred),
                "f1": f1_score(yva, pred, zero_division=0),
                "roc_auc": roc_auc_score(yva, prob),
                "pr_auc": average_precision_score(yva, prob),
                "balanced_accuracy": balanced_accuracy_score(yva, pred),
                "mcc": matthews_corrcoef(yva, pred),
            })

per_fold = pd.DataFrame(rows)
per_fold.to_csv(OUTDIR / "exp12_xgb_10fold_per_fold.csv", index=False)

metric_cols = [
    "accuracy", "precision", "recall", "specificity", "f1",
    "roc_auc", "pr_auc", "balanced_accuracy", "mcc",
]

summary = (
    per_fold
    .groupby(["feature_set", "n_features", "condition"])[metric_cols]
    .agg(["mean", "std"])
)

display(summary)
summary.to_csv(OUTDIR / "exp12_xgb_10fold_summary.csv")


## 8. Interpretation

- **TN12-A (93 features):** multivariable XGBoost on the complete Rao local activation representation.
- **TN12-B (62 features):** multivariable XGBoost restricted to the 62 features listed in Rao Table 2.

For the closest Rao-oriented comparison, use `None` as the primary weighting condition. `Balanced` is a supplementary thesis-oriented condition.

These are MGMT patient-level results and must not be directly ranked against the thesis Tumor/Non-tumor voxel-level metrics.
